# Ensemble Sampler (emcee-style)

This notebook demonstrates the affine-invariant ensemble sampler (Goodman & Weare, 2010) in minibayes.

**Key features:**
- **Affine-invariant**: Works well regardless of parameter scaling (no tuning required)
- **Parallel walkers**: K walkers explore the space simultaneously
- **Stretch moves**: Proposals are generated using other walkers' positions

**Examples in this notebook:**
1. **Hierarchical Model**: Eight Schools problem with partial pooling
2. **MVN with Unknown Covariance**: Comparing Ensemble vs Adaptive MH

## Algorithm Overview

The ensemble sampler uses K walkers that move according to the **stretch move**:
- For walker j at position $x_j$, select random walker k from complementary set
- Proposal: $y = x_k + z \cdot (x_j - x_k)$ where $z$ is drawn from a special distribution
- Accept with probability $\min(1, z^{d-1} \cdot p(y)/p(x_j))$

The **red-blue splitting** pattern ensures walkers don't use their own position in proposals.

**References:**
- Goodman & Weare (2010) "Ensemble samplers with affine invariance"
- Foreman-Mackey et al. (2013) "emcee: The MCMC Hammer"

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

import minibayes as mb
from minibayes import Model, dist, viz
from minibayes.diagnostics import effective_sample_size, summary

---
## Example 1: Hierarchical Model (Eight Schools)

The classic Eight Schools problem (Rubin, 1981): estimating SAT coaching effects across 8 schools.

**Hierarchical Model:**
$$\mu \sim N(0, 5)$$
$$\tau \sim \text{HalfNormal}(5)$$
$$\theta_j \sim N(\mu, \tau) \quad \text{for } j = 1, \ldots, 8$$
$$y_j \sim N(\theta_j, \sigma_j)$$

This model has a **funnel geometry** when $\tau$ is small, which can be challenging for traditional samplers.

In [ ]:
# Eight Schools data
school_names = ["A", "B", "C", "D", "E", "F", "G", "H"]
y = np.array([28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0])  # Estimated effects
sigma = np.array([15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0])  # Standard errors

J = len(y)

print("Eight Schools Data:")
print("-" * 40)
for j, name in enumerate(school_names):
    print(f"School {name}: effect = {y[j]:5.1f} +/- {sigma[j]:.1f}")

In [ ]:
from numpy.typing import NDArray


# Define hierarchical model
def priors(p):
    mu = p("mu", dist.Normal(0, 5))       # Overall mean
    tau = p("tau", dist.HalfNormal(5))    # Between-school std
    theta = p("theta", dist.Normal(mu, tau), size=J)  # School effects


def log_likelihood(params, data) -> NDArray[np.float64]:
    y_obs, sigma_obs = data
    theta = params["theta"]
    # Return pointwise log-likelihood for each school
    ll: NDArray[np.float64] = np.zeros(len(y_obs), dtype=np.float64)
    for j in range(len(y_obs)):
        ll[j] = float(dist.Normal(theta[j], sigma_obs[j]).log_prob(y_obs[j]))
    return ll


model = Model(priors=priors, log_likelihood=log_likelihood)
print(f"Parameters: {model.param_names}")
print(f"Flat parameters: {model.flat_param_names}")

In [ ]:
# Run ensemble sampler
# num_chains = num_walkers for ensemble sampler
# Recommendation: use at least 2 * ndim walkers (here ndim = 10: mu, tau, theta[0-7])

result = mb.sample(
    model,
    data=(y, sigma),
    num_samples=50000,
    num_warmup=1000,
    num_chains=20,  # 24 walkers (>= 2 * 10 = 20)
    sampler="ensemble",
    seed=41,
    progress=True,
)

print(f"\nSampling complete in {result.elapsed_time:.2f}s")
print(f"Acceptance rate: {result.acceptance_rate[0]:.1%}")
print(f"Shape of theta samples: {result.samples['theta'].shape}")
print(f"  -> (num_walkers, num_samples, num_schools)")

In [ ]:
# Summary statistics
print(viz.summary_table(result))

In [ ]:
# Trace plots for hyperparameters (showing multiple walkers)
fig = viz.plot_samples(result, params=["mu", "tau"])
plt.suptitle("Hyperparameter Traces (24 walkers)", y=1.02)
plt.show()

In [ ]:
# Density plots
fig = viz.plot_density(result, params=["mu", "tau"])
plt.show()

In [ ]:
# Shrinkage plot: raw estimates vs posterior estimates
theta_samples = result.samples["theta"]  # (walkers, samples, schools)
theta_mean = theta_samples.mean(axis=(0, 1))
theta_std = theta_samples.std(axis=(0, 1))
mu_mean = result.samples["mu"].mean()

with viz.style():
    fig, ax = plt.subplots(figsize=(10, 6))

    # Raw estimates
    ax.errorbar(range(J), y, yerr=1.96 * sigma, fmt='s', capsize=5,
                color=viz.PALETTE["terracotta"], markersize=10,
                label="Raw estimate (95% CI)", alpha=0.7)

    # Posterior estimates
    ax.errorbar(np.arange(J) + 0.15, theta_mean, yerr=1.96 * theta_std, fmt='o', capsize=5,
                color=viz.PALETTE["blue"], markersize=10,
                label="Posterior (95% CI)")

    # Overall mean
    ax.axhline(mu_mean, color=viz.PALETTE["sage"], linestyle="--", lw=2,
               label=f"Overall mean (mu) = {mu_mean:.1f}")

    # Shrinkage arrows
    for j in range(J):
        ax.annotate("", xy=(j + 0.15, theta_mean[j]), xytext=(j, y[j]),
                    arrowprops=dict(arrowstyle="->", color="gray", alpha=0.5))

    ax.set_xticks(range(J))
    ax.set_xticklabels(school_names)
    ax.set_xlabel("School")
    ax.set_ylabel("Treatment Effect")
    ax.set_title("Shrinkage: Raw Estimates vs Posterior (Ensemble Sampler)")
    ax.legend(loc="upper right")
    plt.show()

print("\nPartial pooling shrinks extreme estimates toward the overall mean.")

---
## Example 2: MVN with Unknown Covariance

Estimate means, standard deviations, and correlation from bivariate data.

**Model:**
- $\mu_1, \mu_2 \sim N(0, 20)$ (means)
- $\sigma_1, \sigma_2 \sim \text{HalfNormal}(10)$ (scales)
- $L_{\text{corr}} \sim \text{LKJ}(\eta=2)$ (correlation Cholesky factor)
- $y_i \sim \text{MVN}(\mu, \Sigma)$ where $\Sigma = D \cdot R \cdot D$ with $D = \text{diag}(\sigma)$, $R = L L^T$

**We'll compare:**
- **Adaptive MH**: Single chain with covariance adaptation
- **Ensemble**: Multiple walkers with stretch moves

In [ ]:
# Generate bivariate data with known correlation
rng = np.random.default_rng(123)

mu_true = np.array([5.0, -3.0])
sigma_true = np.array([2.0, 1.5])
rho_true = 0.7

# Build covariance matrix
corr_true = np.array([[1.0, rho_true], [rho_true, 1.0]])
D = np.diag(sigma_true)
cov_true = D @ corr_true @ D

n_obs = 200
y_obs = rng.multivariate_normal(mu_true, cov_true, size=n_obs)

print(f"True parameters: mu=[{mu_true[0]}, {mu_true[1]}], sigma=[{sigma_true[0]}, {sigma_true[1]}], rho={rho_true}")
print(f"Sample means: [{y_obs[:, 0].mean():.2f}, {y_obs[:, 1].mean():.2f}]")
print(f"Sample correlation: {np.corrcoef(y_obs.T)[0, 1]:.3f}")

In [ ]:
# Visualize data
with viz.style():
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_obs[:, 0], y_obs[:, 1], alpha=0.6, color=viz.PALETTE["blue"], s=40)
    ax.scatter(mu_true[0], mu_true[1], color=viz.PALETTE["terracotta"], s=150,
               marker="*", label=f"True mean ({mu_true[0]}, {mu_true[1]})", zorder=10)
    ax.set_xlabel("$y_1$")
    ax.set_ylabel("$y_2$")
    ax.set_title(f"Bivariate Data (n={n_obs}, $\\rho$={rho_true})")
    ax.legend()
    ax.set_aspect("equal")
    plt.show()

In [ ]:
# Define model with LKJ prior for correlation
def priors_mvn(p):
    p("mu1", dist.Normal(0, 20))
    p("mu2", dist.Normal(0, 20))
    p("sigma1", dist.HalfNormal(10))
    p("sigma2", dist.HalfNormal(10))
    p("L_corr", dist.LKJCholesky(dim=2, eta=2.0), shape=(2, 2))


def log_likelihood_mvn(params, data) -> NDArray[np.float64]:
    mean = np.array([params["mu1"], params["mu2"]])
    L_corr = params["L_corr"]
    corr = L_corr @ L_corr.T
    sigma_diag = np.diag([params["sigma1"], params["sigma2"]])
    cov = sigma_diag @ corr @ sigma_diag
    return dist.MultivariateNormal(mean=mean, cov=cov).log_prob(data)


model_mvn = Model(priors=priors_mvn, log_likelihood=log_likelihood_mvn)
print(f"Parameters: {model_mvn.param_names}")
print(f"Flat parameters: {model_mvn.flat_param_names}")
print(f"Number of unconstrained parameters: {len(model_mvn.flat_param_names)}")

### Comparison: Adaptive MH vs Ensemble (Same Total Samples)

Both samplers will draw **20,000 total samples** for a fair comparison:
- **Adaptive MH**: 4 chains × 5,000 samples
- **Ensemble**: 20 walkers × 1,000 samples

In [ ]:
# Run Adaptive MH (4 chains)
# Total samples: 4 chains × 5000 = 20,000

print("=" * 60)
print("ADAPTIVE METROPOLIS-HASTINGS (4 chains)")
print("=" * 60)

NUM_SAMPLES_AMH = 5000
NUM_CHAINS_AMH = 4

start_amh = time.perf_counter()
result_amh = mb.sample(
    model_mvn,
    data=y_obs,
    num_samples=NUM_SAMPLES_AMH,
    num_warmup=2000,
    num_chains=NUM_CHAINS_AMH,
    sampler="adaptive_mh",
    seed=42,
    progress=True,
)
time_amh = time.perf_counter() - start_amh

total_amh = NUM_CHAINS_AMH * NUM_SAMPLES_AMH
print(f"\nTime: {time_amh:.2f}s")
print(f"Total samples: {NUM_CHAINS_AMH} chains × {NUM_SAMPLES_AMH} = {total_amh:,}")
print(f"Acceptance rates: {[f'{r:.1%}' for r in result_amh.acceptance_rate]}")

In [ ]:
# Run Ensemble sampler
# ndim = 5 (mu1, mu2, sigma1, sigma2, L_corr[1,0] in unconstrained space)
# Total samples: 20 walkers × 1000 = 20,000 (same as AMH)

print("=" * 60)
print("ENSEMBLE SAMPLER (20 walkers)")
print("=" * 60)

NUM_WALKERS = 20  # >= 2 * ndim = 10
NUM_SAMPLES_ENS = 1000

start_ens = time.perf_counter()
result_ens = mb.sample(
    model_mvn,
    data=y_obs,
    num_samples=NUM_SAMPLES_ENS,
    num_warmup=500,
    num_chains=NUM_WALKERS,
    sampler="ensemble",
    seed=42,
    progress=True,
)
time_ens = time.perf_counter() - start_ens

total_ens = NUM_WALKERS * NUM_SAMPLES_ENS
print(f"\nTime: {time_ens:.2f}s")
print(f"Total samples: {NUM_WALKERS} walkers × {NUM_SAMPLES_ENS} = {total_ens:,}")
print(f"Acceptance rate: {result_ens.acceptance_rate[0]:.1%}")

In [ ]:
# Compare results
print("\n" + "=" * 70)
print("COMPARISON: ADAPTIVE MH vs ENSEMBLE (same total samples)")
print("=" * 70)

# Extract correlation from L_corr (rho = L[1,0] for 2x2)
rho_amh = result_amh.samples["L_corr"][:, :, 1, 0].flatten()
rho_ens = result_ens.samples["L_corr"][:, :, 1, 0].flatten()

# Helper function to compute average ESS across chains/walkers
def avg_ess(samples_2d):
    """Compute ESS per chain and average."""
    ess_list = [effective_sample_size(samples_2d[i]) for i in range(samples_2d.shape[0])]
    return np.mean(ess_list)

# Compute ESS for key parameters
ess_amh_mu1 = avg_ess(result_amh.samples["mu1"])
ess_amh_rho = avg_ess(result_amh.samples["L_corr"][:, :, 1, 0])
ess_ens_mu1 = avg_ess(result_ens.samples["mu1"])
ess_ens_rho = avg_ess(result_ens.samples["L_corr"][:, :, 1, 0])

total_amh = result_amh.samples["mu1"].size
total_ens = result_ens.samples["mu1"].size

print(f"\n{'Metric':<25} {'Adaptive MH':>15} {'Ensemble':>15} {'True':>10}")
print("-" * 70)
print(f"{'Total samples':<25} {total_amh:>15,} {total_ens:>15,} {'':>10}")
print(f"{'Time (s)':<25} {time_amh:>15.2f} {time_ens:>15.2f} {'':>10}")
print(f"{'Samples/second':<25} {total_amh/time_amh:>15,.0f} {total_ens/time_ens:>15,.0f} {'':>10}")
print("-" * 70)
print(f"{'mu1 (mean)':<25} {result_amh.samples['mu1'].mean():>15.3f} {result_ens.samples['mu1'].mean():>15.3f} {mu_true[0]:>10.3f}")
print(f"{'mu2 (mean)':<25} {result_amh.samples['mu2'].mean():>15.3f} {result_ens.samples['mu2'].mean():>15.3f} {mu_true[1]:>10.3f}")
print(f"{'sigma1 (mean)':<25} {result_amh.samples['sigma1'].mean():>15.3f} {result_ens.samples['sigma1'].mean():>15.3f} {sigma_true[0]:>10.3f}")
print(f"{'sigma2 (mean)':<25} {result_amh.samples['sigma2'].mean():>15.3f} {result_ens.samples['sigma2'].mean():>15.3f} {sigma_true[1]:>10.3f}")
print(f"{'rho (mean)':<25} {rho_amh.mean():>15.3f} {rho_ens.mean():>15.3f} {rho_true:>10.3f}")
print("-" * 70)
print(f"{'ESS (mu1, avg/chain)':<25} {ess_amh_mu1:>15.0f} {ess_ens_mu1:>15.0f} {'':>10}")
print(f"{'ESS (rho, avg/chain)':<25} {ess_amh_rho:>15.0f} {ess_ens_rho:>15.0f} {'':>10}")
print(f"{'ESS/s (mu1)':<25} {ess_amh_mu1/time_amh:>15.0f} {ess_ens_mu1/time_ens:>15.0f} {'':>10}")

In [ ]:
# Compare trace plots
n_chains_amh = result_amh.samples["mu1"].shape[0]
n_walkers = result_ens.samples["mu1"].shape[0]
n_trace = min(500, result_ens.samples["mu1"].shape[1])  # Show first 500 or all if fewer

with viz.style():
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # AMH: mu1
    ax = axes[0, 0]
    for chain in range(n_chains_amh):
        ax.plot(result_amh.samples["mu1"][chain, :n_trace], alpha=0.7, lw=0.5)
    ax.axhline(mu_true[0], color=viz.PALETTE["terracotta"], ls="--", lw=2)
    ax.set_title(f"Adaptive MH: mu1 (first {n_trace} samples, {n_chains_amh} chains)")
    ax.set_ylabel("mu1")

    # Ensemble: mu1
    ax = axes[0, 1]
    for walker in range(n_walkers):
        ax.plot(result_ens.samples["mu1"][walker, :n_trace], alpha=0.4, lw=0.5)
    ax.axhline(mu_true[0], color=viz.PALETTE["terracotta"], ls="--", lw=2)
    ax.set_title(f"Ensemble: mu1 (first {n_trace} samples, {n_walkers} walkers)")

    # AMH: rho
    ax = axes[1, 0]
    for chain in range(n_chains_amh):
        rho_chain = result_amh.samples["L_corr"][chain, :n_trace, 1, 0]
        ax.plot(rho_chain, alpha=0.7, lw=0.5)
    ax.axhline(rho_true, color=viz.PALETTE["terracotta"], ls="--", lw=2)
    ax.set_title(f"Adaptive MH: rho (first {n_trace} samples)")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("rho")

    # Ensemble: rho
    ax = axes[1, 1]
    for walker in range(n_walkers):
        rho_walker = result_ens.samples["L_corr"][walker, :n_trace, 1, 0]
        ax.plot(rho_walker, alpha=0.4, lw=0.5)
    ax.axhline(rho_true, color=viz.PALETTE["terracotta"], ls="--", lw=2)
    ax.set_title(f"Ensemble: rho (first {n_trace} samples, {n_walkers} walkers)")
    ax.set_xlabel("Iteration")

    plt.tight_layout()
    plt.show()

In [ ]:
# Compare posterior densities
with viz.style():
    fig, axes = plt.subplots(2, 3, figsize=(12, 6))

    params_to_compare = [
        ("mu1", mu_true[0]),
        ("mu2", mu_true[1]),
        ("sigma1", sigma_true[0]),
        ("sigma2", sigma_true[1]),
    ]

    for idx, (param, true_val) in enumerate(params_to_compare):
        ax = axes.flat[idx]

        amh_samples = result_amh.samples[param].flatten()
        ens_samples = result_ens.samples[param].flatten()

        ax.hist(amh_samples, bins=50, density=True, alpha=0.5,
                color=viz.PALETTE["blue"], label="Adaptive MH")
        ax.hist(ens_samples, bins=50, density=True, alpha=0.5,
                color=viz.PALETTE["sage"], label="Ensemble")
        ax.axvline(true_val, color=viz.PALETTE["terracotta"], ls="--", lw=2, label="True")
        ax.set_xlabel(param)
        ax.set_ylabel("Density")
        if idx == 0:
            ax.legend()

    # rho
    ax = axes.flat[4]
    ax.hist(rho_amh, bins=50, density=True, alpha=0.5,
            color=viz.PALETTE["blue"], label="Adaptive MH")
    ax.hist(rho_ens, bins=50, density=True, alpha=0.5,
            color=viz.PALETTE["sage"], label="Ensemble")
    ax.axvline(rho_true, color=viz.PALETTE["terracotta"], ls="--", lw=2, label="True")
    ax.set_xlabel("rho (correlation)")
    ax.set_ylabel("Density")

    # Hide last subplot
    axes.flat[5].axis("off")

    plt.suptitle("Posterior Comparison: Adaptive MH vs Ensemble", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Joint posterior for mu1, mu2
with viz.style():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Adaptive MH
    ax = axes[0]
    ax.scatter(result_amh.samples["mu1"].flatten()[::10],
               result_amh.samples["mu2"].flatten()[::10],
               alpha=0.2, s=10, color=viz.PALETTE["blue"])
    ax.scatter(mu_true[0], mu_true[1], color=viz.PALETTE["terracotta"],
               s=200, marker="*", zorder=10, label="True")
    ax.set_xlabel("mu1")
    ax.set_ylabel("mu2")
    ax.set_title("Adaptive MH: Joint Posterior")
    ax.legend()

    # Ensemble
    ax = axes[1]
    ax.scatter(result_ens.samples["mu1"].flatten()[::10],
               result_ens.samples["mu2"].flatten()[::10],
               alpha=0.1, s=10, color=viz.PALETTE["sage"])
    ax.scatter(mu_true[0], mu_true[1], color=viz.PALETTE["terracotta"],
               s=200, marker="*", zorder=10, label="True")
    ax.set_xlabel("mu1")
    ax.set_ylabel("mu2")
    ax.set_title("Ensemble: Joint Posterior")
    ax.legend()

    plt.tight_layout()
    plt.show()

---
## Summary

### When to Use Ensemble Sampler

| Scenario | Recommended |
|----------|-------------|
| Correlated parameters | Ensemble or Adaptive MH |
| Multimodal posteriors | Ensemble (walkers can explore different modes) |
| No tuning desired | Ensemble (affine-invariant, no proposal tuning) |
| Memory constrained | Adaptive MH (fewer parallel states) |
| Maximizing ESS/s | Depends on problem (test both) |

### Key Differences

| Aspect | Adaptive MH | Ensemble |
|--------|-------------|----------|
| Tuning | Auto-adapts covariance | No tuning (affine-invariant) |
| Parallelism | Independent chains | Walkers interact via stretch moves |
| Memory | O(warmup samples) | O(num_walkers) |
| Acceptance | Typically 20-40% | Typically 30-50% |
| Multimodal | May get stuck | Walkers can span modes |

### Usage

```python
# Ensemble sampler
result = mb.sample(
    model, data,
    num_samples=5000,
    num_warmup=1000,
    num_chains=24,  # = num_walkers (use >= 2 * ndim)
    sampler="ensemble",
    sampler_kwargs={"stretch_scale": 2.0},  # optional (default=2.0)
)
```

Each walker produces a chain of samples. The `num_chains` parameter controls the number of walkers.